In [ ]:
!pip install -q gradio

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import gradio as gr
import tensorflow as tf
import numpy as np
from PIL import Image
from tensorflow.keras.applications.resnet50 import preprocess_input

In [ ]:
MODEL_PATH = "/content/drive/MyDrive/Ayurvedic_project/model.h5"
model = tf.keras.models.load_model(MODEL_PATH)

In [ ]:
class_names = ['Amruta_Balli', 'Ashwagandha', 'Bhrami', 'Ekka',
               'Hibiscus', 'Insulin', 'Money plant',
               'Parijata', 'Sandalwood', 'Tulsi', 'Turmeric']

In [ ]:
AYURVEDIC_PLANTS = [
    'Amruta_Balli', 'Ashwagandha', 'Bhrami',
    'Ekka', 'Hibiscus', 'Insulin',
    'Parijata', 'Sandalwood', 'Tulsi', 'Turmeric'
]

In [ ]:
PLANT_INFO = {
    "Amruta_Balli": {"en": "Boosts immunity.", "kn": "ರೋಗನಿರೋಧಕ ಶಕ್ತಿ ಹೆಚ್ಚಿಸುತ್ತದೆ."},
    "Ashwagandha": {"en": "Reduces stress.", "kn": "ಒತ್ತಡ ಕಡಿಮೆ ಮಾಡುತ್ತದೆ."},
    "Bhrami": {"en": "Improves memory.", "kn": "ಸ್ಮರಣೆ ಸುಧಾರಿಸುತ್ತದೆ."},
    "Ekka": {"en": "Used for skin problems.", "kn": "ಚರ್ಮಕ್ಕೆ ಉಪಯುಕ್ತ."},
    "Hibiscus": {"en": "Supports hair growth.", "kn": "ಕೂದಲು ಬೆಳವಣಿಗೆಗೆ ಸಹಾಯಕ."},
    "Insulin": {"en": "Controls blood sugar.", "kn": "ಸಕ್ಕರೆ ನಿಯಂತ್ರಿಸುತ್ತದೆ."},
    "Money plant": {
        "en": "Not an Ayurvedic plant, used for decoration.",
        "kn": "ಆಯುರ್ವೇದ ಸಸ್ಯವಲ್ಲ, ಅಲಂಕಾರಕ್ಕೆ ಬಳಸಲಾಗುತ್ತದೆ."
    },
    "Parijata": {"en": "Used for fever.", "kn": "ಜ್ವರಕ್ಕೆ ಉಪಯುಕ್ತ."},
    "Sandalwood": {"en": "Used for skin care.", "kn": "ಚರ್ಮದ ಆರೈಕೆ."},
    "Tulsi": {"en": "Improves immunity.", "kn": "ರೋಗನಿರೋಧಕ ಶಕ್ತಿ ಹೆಚ್ಚಿಸುತ್ತದೆ."},
    "Turmeric": {"en": "Healing properties.", "kn": "ಗುಣಪಡಿಸುವ ಗುಣಗಳು."}
}

In [ ]:
def remove_background_green(img):
    r, g, b = tf.unstack(img, axis=-1)

    exg = 2.0 * g - r - b

    exg_min = tf.reduce_min(exg)
    exg_max = tf.reduce_max(exg)

    exg_norm = (exg - exg_min) / (exg_max - exg_min + 1e-5)

    mask = exg_norm > 0.4
    mask = tf.cast(mask, tf.float32)
    mask = tf.expand_dims(mask, axis=-1)

    return img * mask

In [ ]:
def preprocess(img):

    img = img.resize((224, 224))

    img = np.array(img).astype(np.float32) / 255.0

    img = tf.convert_to_tensor(img)

    img = remove_background_green(img)

    img = img.numpy()

    img = img * 255.0

    img = preprocess_input(img)

    img = np.expand_dims(img, axis=0)

    return img

In [ ]:
def predict(image):

    if image is None:
        return "No image", "", "", None

    img = preprocess(image)

    preds = model.predict(img, verbose=0)[0]

    print("\nPrediction probabilities:")
    for i, name in enumerate(class_names):
        print(f"{name}: {preds[i]:.4f}")

    best_idx = np.argmax(preds)
    confidence = float(preds[best_idx])

    plant = class_names[best_idx]

    # Lower threshold for testing
    if confidence < 0.40:
        return "This is not a plant", "", "", image

    is_ayurvedic = "Yes" if plant in AYURVEDIC_PLANTS else "No"

    info_en = PLANT_INFO[plant]["en"]
    info_kn = PLANT_INFO[plant]["kn"]

    description = (
        f"Confidence : {confidence*100:.2f}%\n\n"
        f"🌐 English: {info_en}\n"
        f"🇮🇳 Kannada: {info_kn}"
    )

    return plant, is_ayurvedic, description, image

In [ ]:
with gr.Blocks() as demo:

    # 🔥 BIG BOLD TITLE
    gr.Markdown(
        "<h1 style='text-align:center; font-size:38px; font-weight:bold;'>🌿 Ayurvedic Plant Identification</h1>"
    )

    gr.Markdown("### Upload image or capture using camera")

    # ✅ UPDATED CAMERA + UPLOAD (LATEST GRADIO)
    image_input = gr.Image(
        sources=["upload", "webcam"],
        type="pil",
        label="Upload / Capture Image"
    )

    btn = gr.Button("Predict")

    plant_out = gr.Textbox(label="Plant Name")
    ayur_out = gr.Textbox(label="Ayurvedic?")
    desc_out = gr.Textbox(label="Description")
    img_out = gr.Image(label="Preview")

    btn.click(
        predict,
        inputs=image_input,
        outputs=[plant_out, ayur_out, desc_out, img_out]
    )

In [ ]:
demo.launch(debug=True, share=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://99776695d7ec75372d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



Prediction probabilities:
Amruta_Balli: 0.0337
Ashwagandha: 0.0065
Bhrami: 0.0003
Ekka: 0.8858
Hibiscus: 0.0046
Insulin: 0.0217
Money plant: 0.0004
Parijata: 0.0445
Sandalwood: 0.0025
Tulsi: 0.0000
Turmeric: 0.0000

Prediction probabilities:
Amruta_Balli: 0.6683
Ashwagandha: 0.1663
Bhrami: 0.0112
Ekka: 0.0810
Hibiscus: 0.0284
Insulin: 0.0004
Money plant: 0.0138
Parijata: 0.0263
Sandalwood: 0.0022
Tulsi: 0.0012
Turmeric: 0.0008

Prediction probabilities:
Amruta_Balli: 0.0472
Ashwagandha: 0.0008
Bhrami: 0.0001
Ekka: 0.0005
Hibiscus: 0.9407
Insulin: 0.0000
Money plant: 0.0000
Parijata: 0.0107
Sandalwood: 0.0000
Tulsi: 0.0001
Turmeric: 0.0000

Prediction probabilities:
Amruta_Balli: 0.0033
Ashwagandha: 0.4577
Bhrami: 0.0001
Ekka: 0.0054
Hibiscus: 0.4278
Insulin: 0.0001
Money plant: 0.0003
Parijata: 0.0355
Sandalwood: 0.0326
Tulsi: 0.0001
Turmeric: 0.0372

Prediction probabilities:
Amruta_Balli: 0.0010
Ashwagandha: 0.0009
Bhrami: 0.9563
Ekka: 0.0001
Hibiscus: 0.0104
Insulin: 0.0001
Money p